# Tono · el arreglo: calibrar el umbral por banda de tono

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

El experimento anterior encontró que el modelo **discrimina igual de bien en
piel oscura** (AUROC 0,910 frente a 0,900 en clara) pero al umbral compartido
**se pierde el 23,7% de los cánceres** frente al 10-15% en las otras bandas.
Mismo poder, peor punto de operación: es calibración, no discriminación.

Aquí se corrige, con dos condiciones que hacen el resultado creíble:

1. **Los umbrales se derivan en VALIDACIÓN y se aplican a test.** Ajustarlos
   sobre test daría una mejora que no existiría al desplegar el modelo.
2. **Tres semillas.** La banda oscura del test tiene ~69 imágenes; sin repetir,
   no se distingue una mejora real del ruido de inicialización.

Se comparan tres estrategias, todas calibradas en validación:

| Estrategia | Idea |
|---|---|
| Umbral único | lo que había |
| Youden por grupo | maximiza sensibilidad+especificidad en cada banda |
| **Sensibilidad fija por grupo** | **iguala el daño**: se fija cuántos casos se está dispuesto a perder, igual para todos |

La tercera es la defendible en cribado de cáncer, porque iguala lo que le pasa
a las personas en lugar de igualar un número abstracto.

Además se entrena con **regularización más fuerte**: la versión anterior llegaba
a un AUROC de entrenamiento de 0,99998 y un modelo que memoriza da estimaciones
por subgrupo más ruidosas.

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', '/tmp/' + nombre], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/tmp/' + nombre], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert 'sm_%d%d' % cap in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')

In [ ]:
OUT = '/kaggle/working'
os.chdir('/tmp/tono')
anclas = glob.glob('/kaggle/input/**/fitzpatrick17k*.csv', recursive=True)
FITZ = os.path.dirname(anclas[0])
print('Fitzpatrick17k en:', FITZ)

!python -m datos.fitzpatrick17k --root {FITZ} --out {OUT}/manifiesto_fitz.csv

## 1. Tres semillas con regularización fuerte

In [ ]:
SEMILLAS = [42, 1337, 2024]
os.chdir('/tmp/cxr')

for s in SEMILLAS:
    print('===== semilla %d =====' % s, flush=True)
    !python -m src.train --config /tmp/tono/configs/fitzpatrick17k_reg.yaml --manifest {OUT}/manifiesto_fitz.csv --out-dir {OUT}/runs/s{s} --seed {s}
    # Validacion: de aqui salen los umbrales. Test: donde se aplican.
    !python -m src.evaluate --checkpoint {OUT}/runs/s{s}/best.pth --manifest {OUT}/manifiesto_fitz.csv --split val --out-dir {OUT}/reports/s{s}_val --n-boot 200
    !python -m src.evaluate --checkpoint {OUT}/runs/s{s}/best.pth --manifest {OUT}/manifiesto_fitz.csv --split test --out-dir {OUT}/reports/s{s}_test --n-boot 500

In [ ]:
import pandas as pd, numpy as np

print('Sobreajuste: train vs val AUROC en la ultima epoca de cada semilla')
for s in SEMILLAS:
    h = pd.read_csv(OUT + '/runs/s%d/history.csv' % s)
    u = h.iloc[-1]
    print('  semilla %-5d epocas=%-3d train %.4f  val %.4f  (antes: 0.99998 / 0.900)'
          % (s, len(h), u.train_auroc, u.val_auroc))

aurocs = []
for s in SEMILLAS:
    m = json.load(open(OUT + '/reports/s%d_test/metrics.json' % s))
    aurocs.append(m['auroc'])
    print('  semilla %-5d AUROC test %.4f' % (s, m['auroc']))
print('AUROC medio %.4f +- %.4f' % (float(np.mean(aurocs)), float(np.std(aurocs, ddof=1))))

## 2. El arreglo: umbrales derivados en validación

Para cada semilla se calibran las tres estrategias sobre el conjunto de
validación y se aplican al de test, que el proceso de calibración nunca ve.

In [ ]:
sys.path.insert(0, '/tmp/tono')
sys.path.insert(0, '/tmp')
from scripts.calibrar_por_grupo import calibrar

resultados = {}
for s in SEMILLAS:
    val = pd.read_csv(OUT + '/reports/s%d_val/predictions.csv' % s)
    test = pd.read_csv(OUT + '/reports/s%d_test/predictions.csv' % s)
    resultados[s] = calibrar(val, test, grupo='fitzpatrick',
                             sensibilidad_objetivo=0.85, min_n=25)
    print('semilla', s, 'calibrada')

In [ ]:
nombres = list(resultados[SEMILLAS[0]]['estrategias'].keys())
resumen_estrategias = {}

print('BRECHA DE FNR ENTRE BANDAS DE TONO, media de 3 semillas')
print('=' * 62)
for nombre in nombres:
    brechas = [resultados[s]['estrategias'][nombre]['brecha_fnr'] for s in SEMILLAS]
    oscura = []
    for s in SEMILLAS:
        for fila in resultados[s]['estrategias'][nombre]['tabla']:
            if 'oscura' in fila['grupo']:
                oscura.append(fila['fnr'])
    resumen_estrategias[nombre] = {
        'brecha_fnr_media': round(float(np.mean(brechas)), 4),
        'brecha_fnr_desv': round(float(np.std(brechas, ddof=1)), 4),
        'fnr_piel_oscura_media': round(float(np.mean(oscura)), 4) if oscura else None,
        'brechas_por_semilla': brechas,
    }
    o = resumen_estrategias[nombre]['fnr_piel_oscura_media']
    print('%-34s brecha %.4f +- %.4f   FNR piel oscura %s'
          % (nombre, float(np.mean(brechas)), float(np.std(brechas, ddof=1)),
             ('%.4f' % o) if o is not None else 'n/d'))

In [ ]:
print('Detalle por banda, semilla 42')
for nombre in nombres:
    print()
    print('---', nombre, '---')
    print(pd.DataFrame(resultados[42]['estrategias'][nombre]['tabla']).to_string(index=False))

## 3. Resumen

In [ ]:
mejor = min(resumen_estrategias.items(), key=lambda kv: kv[1]['brecha_fnr_media'])
resumen = {
    'problema': 'mismo AUROC en piel oscura pero FNR 0,237 frente a 0,103-0,152 al umbral unico',
    'metodo': 'umbrales derivados en validacion y aplicados a test; 3 semillas',
    'semillas': SEMILLAS,
    'auroc_test_medio': round(float(np.mean(aurocs)), 4),
    'auroc_test_desv': round(float(np.std(aurocs, ddof=1)), 4),
    'estrategias': resumen_estrategias,
    'mejor_estrategia': mejor[0],
    'detalle_semilla_42': resultados[42],
    'minutos': round((time.time() - T0) / 60, 1),
}
json.dump(resumen, open(OUT + '/resumen.json', 'w'), indent=2, ensure_ascii=False)
print('Mejor estrategia:', mejor[0], '-> brecha', mejor[1]['brecha_fnr_media'])
print(json.dumps(resumen_estrategias, indent=2, ensure_ascii=False))

import shutil
for f_ in glob.glob(OUT + '/manifiesto_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))
for f_ in glob.glob(OUT + '/runs/*/best.pth'):
    if 's42' not in f_:
        os.remove(f_)